# 02 - Exploratory analysis: what drives the approval decision?

Uses the cleaned primary dataset (`data/processed/loan_approval_clean.csv`, built in `01_audit`). The target is the **approval decision** (1 = Approved, 0 = Rejected) in public synthetic data. Nothing here describes default risk or any real lender's policy.

Charts come from `src/loanlens/charts.py` so the Analytics page can reuse them later. Every claim about the data in CLAUDE.md section 3 is checked against the actual numbers in section 9.

In [1]:
import pandas as pd
import plotly.io as pio
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

from loanlens import charts, cleaning

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
SEED = 42

df = pd.read_csv(cleaning.PROCESSED_PATH)
ASSETS = list(cleaning.ASSET_COLUMNS)
# Derived quantities, computed here for analysis only (features.py comes in a later phase).
lti = (df["loan_amount"] / df["income_annum"]).rename("loan_to_income")
total_assets = df[ASSETS].sum(axis=1).rename("total_assets")
band = charts.cibil_band(df["cibil_score"]).rename("cibil_band")
print(f"{len(df):,} applications, overall approval rate {df.loan_status.mean():.1%}")

4,269 applications, overall approval rate 62.2%


## 1. CIBIL score

In [2]:
charts.approval_by_cibil_band(df)

In [3]:
charts.approval_rate_table(df, band)

,cibil_band,applications,approved,approval_rate
0,300-450,1053,116,0.110
1,451-550,736,73,0.099
2,551-650,688,686,0.997
3,651-750,739,734,0.993
4,751-900,1053,1047,0.994


In [4]:
charts.distribution_by_status(df, "cibil_score", "CIBIL score by approval status")

## 2. Loan term

In [5]:
charts.approval_rate_bar(df, "loan_term", "Approval rate by loan term", "Loan term (years)")

In [6]:
charts.approval_heatmap(
    df,
    band,
    "loan_term",
    "Approval rate by CIBIL band and loan term",
    row_order=list(charts.CIBIL_LABELS),
)

## 3. Loan-to-income ratio

In [7]:
charts.approval_by_quantile_band(
    df, "loan_to_income", q=5, values=lti, title="Approval rate by loan-to-income ratio (quintiles)"
)

In [8]:
charts.distribution_by_status(df, "loan_to_income", "Loan-to-income ratio by status", values=lti)

## 4. Income and loan amount

In [9]:
charts.approval_by_quantile_band(
    df, "income_annum", q=5, title="Approval rate by annual income (quintiles, INR)"
)

In [10]:
charts.approval_by_quantile_band(
    df, "loan_amount", q=5, title="Approval rate by loan amount (quintiles, INR)"
)

## 5. Assets

Each asset type, then the total.

In [11]:
for col in ASSETS:
    charts.approval_by_quantile_band(
        df, col, q=5, title=f"Approval rate by {col} (quintiles, INR)"
    ).show()

In [12]:
charts.approval_by_quantile_band(
    df,
    "total_assets",
    q=5,
    values=total_assets,
    title="Approval rate by total assets (quintiles, INR)",
)

## 6. Education, self-employment, dependents

In [13]:
charts.approval_rate_bar(df, "education", "Approval rate by education")

In [14]:
charts.approval_rate_bar(df, "self_employed", "Approval rate by self-employment")

In [15]:
charts.approval_rate_bar(
    df, "no_of_dependents", "Approval rate by number of dependents", "Dependents"
)

## 7. Statistical check for categorical variables

Chi-square test of independence against the approval decision. A large p-value means no evidence of an effect.

In [16]:
rows = []
for col in ["education", "self_employed", "no_of_dependents", "loan_term"]:
    table = pd.crosstab(df[col], df["loan_status"])
    chi2, p, dof, _ = chi2_contingency(table)
    rates = df.groupby(col)["loan_status"].mean()
    rows.append(
        {
            "variable": col,
            "chi2": chi2,
            "dof": dof,
            "p_value": p,
            "min_rate": rates.min(),
            "max_rate": rates.max(),
            "spread_pct_points": 100 * (rates.max() - rates.min()),
        }
    )
chi_table = pd.DataFrame(rows)
chi_table

,variable,chi2,dof,p_value,min_rate,max_rate,spread_pct_points
0,education,0.084,1,0.772,0.620,0.625,0.477
1,self_employed,0.000,1,1.000,0.622,0.622,0.033
2,no_of_dependents,2.454,5,0.783,0.603,0.642,3.858
3,loan_term,153.533,9,0.000,0.525,0.819,29.356


## 8. Correlation with the target (numeric columns)

In [17]:
numeric = df.select_dtypes("number").assign(loan_to_income=lti, total_assets=total_assets)
numeric.corr()["loan_status"].drop("loan_status").sort_values(key=abs, ascending=False).to_frame(
    "corr_with_approval"
)

,corr_with_approval
cibil_score,0.771
loan_term,-0.113
loan_to_income,0.088
no_of_dependents,-0.018
loan_amount,0.016
luxury_assets_value,-0.015
income_annum,-0.015
residential_assets_value,-0.014
total_assets,-0.011
commercial_assets_value,0.008


## 9. Verifying the CLAUDE.md section 3 claims

Each check prints the actual numbers.

### 9.1 Overall approval share (claim: about 62%)

In [18]:
print(f"Approval share: {df.loan_status.mean():.2%} ({df.loan_status.sum():,} of {len(df):,})")

Approval share: 62.22% (2,656 of 4,269)


### 9.2 CIBIL dominance (claim: about 99% above 550, about 10% at or below 550)

In [19]:
above = df["cibil_score"] > 550
for label, mask in [("CIBIL > 550", above), ("CIBIL <= 550", ~above)]:
    sub = df.loc[mask, "loan_status"]
    print(f"{label:13s}: {sub.mean():.2%} approved ({sub.sum():,} of {len(sub):,})")

# Where is the sharpest boundary? Fit a depth-1 tree on CIBIL only.
stump = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(
    df[["cibil_score"]], df["loan_status"]
)
print(f"Best single CIBIL split learned by a stump: {stump.tree_.threshold[0]:.1f}")

CIBIL > 550  : 99.48% approved (2,467 of 2,480)
CIBIL <= 550 : 10.56% approved (189 of 1,789)
Best single CIBIL split learned by a stump: 549.5


### 9.3 Accuracy claims (claim: CIBIL-only tree about 95%, random forest about 99.9%)

This is only a check on the claim, using stratified 5-fold cross-validation. The proper model comparison is a later phase.

In [20]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
y = df["loan_status"]
X_all = pd.get_dummies(df.drop(columns="loan_status"), drop_first=True)
checks = {
    "Tree, CIBIL only (depth 1)": (
        DecisionTreeClassifier(max_depth=1, random_state=SEED),
        df[["cibil_score"]],
    ),
    "Tree, CIBIL only (unrestricted)": (
        DecisionTreeClassifier(random_state=SEED),
        df[["cibil_score"]],
    ),
    "Random forest, all features": (
        RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        X_all,
    ),
    "Random forest, all + loan_to_income": (
        RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
        X_all.assign(loan_to_income=lti),
    ),
}
acc = {}
for name, (model, X) in checks.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    acc[name] = scores
    print(f"{name:34s}: mean accuracy {scores.mean():.4f} (std {scores.std():.4f})")

Tree, CIBIL only (depth 1)        : mean accuracy 0.9536 (std 0.0052)
Tree, CIBIL only (unrestricted)   : mean accuracy 0.9506 (std 0.0066)


Random forest, all features       : mean accuracy 0.9822 (std 0.0039)


Random forest, all + loan_to_income: mean accuracy 0.9963 (std 0.0020)


### 9.4 Education, self-employment, dependents (claim: no effect)

In [21]:
chi_table.set_index("variable").loc[
    ["education", "self_employed", "no_of_dependents"],
    ["min_rate", "max_rate", "spread_pct_points", "p_value"],
]

,min_rate,max_rate,spread_pct_points,p_value
variable,,,,
education,0.620,0.625,0.477,0.772
self_employed,0.622,0.622,0.033,1.000
no_of_dependents,0.603,0.642,3.858,0.783


### 9.5 Low-CIBIL approvals and loan term (claim: almost all have a 2 or 4 year term)

In [22]:
low = df[df["cibil_score"] <= 550]
low_appr = low[low["loan_status"] == 1]
short = low_appr["loan_term"].isin([2, 4])
print(f"Low-CIBIL (<= 550) applications: {len(low):,}; approved: {len(low_appr):,}")
print(f"Approved with a 2 or 4 year term: {short.sum()} of {len(low_appr)} ({short.mean():.1%})")
print("\nLow-CIBIL approvals by term:", low_appr["loan_term"].value_counts().sort_index().to_dict())
print("\nApproval rate among low-CIBIL applicants by term:")
print(
    low.groupby("loan_term")["loan_status"]
    .agg(["mean", "size"])
    .rename(columns={"mean": "approval_rate", "size": "applications"})
)

Low-CIBIL (<= 550) applications: 1,789; approved: 189
Approved with a 2 or 4 year term: 186 of 189 (98.4%)

Low-CIBIL approvals by term: {2: 82, 4: 104, 10: 1, 16: 1, 20: 1}

Approval rate among low-CIBIL applicants by term:
           approval_rate  applications
loan_term                             
2                  0.480           171
4                  0.568           183
6                  0.000           205
8                  0.000           166
10                 0.005           205
12                 0.000           180
14                 0.000           164
16                 0.006           176
18                 0.000           164
20                 0.006           175


### 9.5b Extra check: loan-to-income inside the low-CIBIL, short-term group

The random forest gains about 1.4 points of accuracy once loan-to-income is added (9.3). This table shows where that comes from.

In [23]:
short_low = df[(df["cibil_score"] <= 550) & df["loan_term"].isin([2, 4])]
lti_band = pd.cut(lti[short_low.index], [1.5, 2.5, 3.0, 3.5, 4.0], include_lowest=True).rename(
    "loan_to_income_band"
)
short_low.groupby(["loan_term", lti_band], observed=True)["loan_status"].agg(
    approval_rate="mean", applications="size"
)

approval_rate  applications
loan_term loan_to_income_band                             
2         (1.499, 2.5]                 0.000            55
          (2.5, 3.0]                   0.083            36
          (3.0, 3.5]                   1.000            44
          (3.5, 4.0]                   0.972            36
4         (1.499, 2.5]                 0.000            42
          (2.5, 3.0]                   0.125            40
          (3.0, 3.5]                   1.000            46
          (3.5, 4.0]                   0.964            55

In [24]:
high_rej = df[(df["cibil_score"] > 550) & (df["loan_status"] == 0)]
print(
    f"Rejections with CIBIL > 550: {len(high_rej)}; loan-to-income median "
    f"{lti[high_rej.index].median():.2f} vs {lti.median():.2f} overall"
)

Rejections with CIBIL > 550: 13; loan-to-income median 3.78 vs 3.00 overall


### 9.6 Loan-to-income range (claim: 1.5 to 4.0)

In [25]:
print(lti.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_string())
print(f"\nRange: {lti.min():.2f} to {lti.max():.2f}")

count   4,269.000
mean        2.985
std         0.595
min         1.500
1%          1.988
5%          2.062
50%         3.000
95%         3.900
99%         4.000
max         4.000

Range: 1.50 to 4.00


### 9.7 Gender and marital status (rule: never used)

In [26]:
banned = [c for c in df.columns if any(k in c.lower() for k in ("gender", "sex", "marri"))]
print(f'Gender / marital columns in the primary dataset: {banned or "none"}')

Gender / marital columns in the primary dataset: none


### 9.8 Summary of claim checks

In [27]:
low_rate = df.loc[~above, "loan_status"].mean()
high_rate = df.loc[above, "loan_status"].mean()
claims = pd.DataFrame(
    [
        ("About 62% approved", f"{df.loan_status.mean():.1%}"),
        ("About 99% approved above CIBIL 550", f"{high_rate:.1%}"),
        ("About 10% approved at or below CIBIL 550", f"{low_rate:.1%}"),
        (
            "CIBIL-only tree about 95% accuracy",
            f"{acc['Tree, CIBIL only (depth 1)'].mean():.1%} (depth 1)",
        ),
        (
            "Random forest about 99.9% accuracy",
            f"{acc['Random forest, all features'].mean():.2%} raw features; "
            f"{acc['Random forest, all + loan_to_income'].mean():.2%} with loan_to_income",
        ),
        (
            "Education / self-employed / dependents: no effect",
            ", ".join(f"{r.variable} p={r.p_value:.2f}" for r in chi_table.iloc[:3].itertuples()),
        ),
        ("Low-CIBIL approvals mostly 2 or 4 year terms", f"{short.mean():.1%}"),
        ("Loan-to-income 1.5 to 4.0", f"{lti.min():.2f} to {lti.max():.2f}"),
    ],
    columns=["claim", "actual"],
)
claims

,claim,actual
0,About 62% approved,62.2%
1,About 99% approved above CIBIL 550,99.5%
2,About 10% approved at or below CIBIL 550,10.6%
3,CIBIL-only tree about 95% accuracy,95.4% (depth 1)
4,Random forest about 99.9% accuracy,98.22% raw features; 99.63% with loan_to_income
5,Education / self-employed / dependents: no effect,"education p=0.77, self_employed p=1.00, no_of_..."
6,Low-CIBIL approvals mostly 2 or 4 year terms,98.4%
7,Loan-to-income 1.5 to 4.0,1.50 to 4.00


## Findings and data limitations

**What drives approval in this dataset**

1. **The CIBIL score decides almost everything.** Above 550, 99.5% of applications are approved. At or below 550, only 10.6% are. A one-question rule ("is CIBIL above 549.5?") is right about 95% of the time.
2. **Loan term matters only for low-CIBIL applicants.** 2 and 4 year loans are approved 78 to 82% of the time, against 53 to 61% for longer terms. That gap comes entirely from the low-CIBIL group: 186 of its 189 approvals (98.4%) have a 2 or 4 year term.
3. **Inside that low-CIBIL, short-term group, a *higher* loan-to-income ratio goes with approval.** Above 3.0, about 97 to 100% are approved. At or below 2.5, none are. This runs against normal lending logic, where a bigger loan relative to income is riskier. It is a quirk of how the synthetic data was generated. Adding loan-to-income lifts random-forest accuracy from 98.2% to 99.6%.
4. **Income, loan amount and assets show no clear effect on their own.** Across quintiles, approval rates stay within about 58 to 68%, and every correlation with approval is below 0.02 in size, except loan-to-income at 0.09.
5. **Education, self-employment and number of dependents make no difference.** The spreads are 0.5, 0.03 and 3.9 percentage points, and the chi-square p-values are 0.77, 1.00 and 0.78.

**Limitations to carry into modelling and the app**

- **Synthetic data.** These patterns describe how the dataset was generated, not any real lender's policy or real borrower behaviour. No real-world accuracy can be claimed.
- **Approval, not default.** There is no repayment data, so nothing here says whether an approved loan would be repaid.
- **Near-perfect scores are expected.** The decision is close to a deterministic rule on CIBIL, term and loan-to-income. High accuracy says the dataset is simple, not that the model is good. The leakage check belongs in the modelling phase.
- **The short-term and loan-to-income quirks must not be shown as advice.** For example, "take a 2-year loan" or "borrow more relative to income" must not appear as suggestions. The "what would change" search will need guarding against this.
- **Narrow ranges.** Loan-to-income only runs from 1.50 to 4.00, loan term from 2 to 20 years (even numbers only), and income from INR 2,00,000 to 99,00,000. Predictions outside these ranges are unreliable and the app should warn.
- **Data errors.** 28 rows had residential assets of exactly -1,00,000. They were set to 0 during cleaning (see `01_audit`).
- **No protected attributes.** The primary dataset has no gender or marital-status columns, and none are used.
